这个代码2025年10月重启EPFLUX计算的测试代码，只使用ERA5数据进行计算看计算程序是否正确。

In [ ]:
# --- EP-flux: compute & cache (wave 1 and 2, 100–1 hPa) ---

import os
import numpy as np
import xarray as xr
import aostools_functions as ac

# 输入/输出
era5_file = "/home/weiji/restart_exam/era5_data/Era5_uvt_2015to2024.nc"
cache_dir = "/home/weiji/restart_exam/plots/epflux_speedtest/cache"
os.makedirs(cache_dir, exist_ok=True)
cache_npz = os.path.join(cache_dir, "epflux_mean_100to1_w1w2.npz")

# 读取并统一坐标；仅保留 100–1 hPa
ds = (
    xr.open_dataset(era5_file)
      .rename({"valid_time":"time", "pressure_level":"plev", "latitude":"lat", "longitude":"lon"})
      .sel(plev=slice(100.0, 1.0))
)


lat  = ds.lat.values
plev = ds.plev.values
u    = ds.u.values
v    = ds.v.values
t    = ds.t.values

# 逐波计算，立刻做时间平均，减少内存
def _compute_time_mean_for_wave(wave_k: int):
    ep1_k, ep2_k, *_ = ac.ComputeEPfluxDiv(lat, plev, u, v, t, wave=wave_k, do_ubar=True)
    ep1_mean = np.nanmean(ep1_k, axis=0)  # (plev, lat)
    ep2_mean = np.nanmean(ep2_k, axis=0)  # (plev, lat)
    return ep1_mean, ep2_mean

ep1_w1, ep2_w1 = _compute_time_mean_for_wave(1)
ep1_w2, ep2_w2 = _compute_time_mean_for_wave(2)

# 保存缓存（注意：存的是 (plev, lat) 排列，绘图时会转成 (lat, plev)）
np.savez_compressed(
    cache_npz,
    lat=lat, plev=plev,
    ep1_w1=ep1_w1, ep2_w1=ep2_w1,
    ep1_w2=ep1_w2, ep2_w2=ep2_w2
)

print(f"✅ Cached: {cache_npz}")
print(f"lat size={lat.size}, plev size={plev.size} (range {plev.max()}–{plev.min()} hPa)")


In [ ]:
import matplotlib.pyplot as plt

cache_npz = "/home/weiji/restart_exam/plots/epflux_speedtest/cache/epflux_mean_100to1_w1w2.npz"
outdir    = "/home/weiji/restart_exam/plots/epflux_speedtest/cache"
os.makedirs(outdir, exist_ok=True)

# 选择绘制的模式：'w1' | 'w2' | 'sum'
mode = "sum"

# 可选的下采样控制（仅影响显示密度，不改变结果）
step_lat  = 15   # 1 表示不稀疏；想降密就设 2、3、4…
step_plev = 1   # 竖直方向通常比较稀，这里也先不下采样

# 读取缓存
Z = np.load(cache_npz)
lat  = Z["lat"]
plev = Z["plev"]
ep1_w1, ep2_w1 = Z["ep1_w1"], Z["ep2_w1"]   # (plev, lat)
ep1_w2, ep2_w2 = Z["ep1_w2"], Z["ep2_w2"]

# 选择要画的场
if mode == "w1":
    ep1, ep2 = ep1_w1, ep2_w1
    tag = "wave1"
elif mode == "w2":
    ep1, ep2 = ep1_w2, ep2_w2
    tag = "wave2"
else:  # 'sum'
    ep1, ep2 = ep1_w1 + ep1_w2, ep2_w1 + ep2_w2
    tag = "wave1_2"

# 组织为 (lat, plev) 以适配 PlotEPfluxArrows
f1 = xr.DataArray(ep1, coords={"plev": plev, "lat": lat}, dims=["plev", "lat"]).transpose("lat","plev")
f2 = xr.DataArray(ep2, coords={"plev": plev, "lat": lat}, dims=["plev", "lat"]).transpose("lat","plev")

# （推荐）去掉最底层，避免近地面极端矢量影响展示
max_plev = float(f1.plev.max())  # 最大 hPa 即最底层
f1p = f1.sel(plev=f1.plev.where(f1.plev < max_plev, drop=True))
f2p = f2.sel(plev=f2.plev.where(f2.plev < max_plev, drop=True))

# 可选：横向/竖向稀疏（只影响显示）
f1p = f1p.isel(lat=slice(None,None,step_lat), plev=slice(None,None,step_plev))
f2p = f2p.isel(lat=slice(None,None,step_lat), plev=slice(None,None,step_plev))

# 绘图：100–1 hPa，log-p
fig, ax = plt.subplots(figsize=(8,6))
ac.PlotEPfluxArrows(
    f1p.lat, f1p.plev,
    f1p, f2p,
    fig, ax,
    xlim=[-90, 90],
    ylim=[100, 1],
    yscale="log",
    scale=1e16  # 需要可视化更清晰可调这个值
)
ax.set_xlabel("Latitude (°)")
ax.set_ylabel("Pressure (hPa)")
ax.set_title(f"EP-Flux ({tag})  ERA5 2015–2024  100–1 hPa  do_ubar=True")

outfile = os.path.join(outdir, f"epflux_{tag}_ERA5_2015to2024_100to1hPa.png")
fig.savefig(outfile, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {outfile}")


In [ ]:
cache_npz = os.path.join(cache_dir, "epflux_div_mean_100to1_w1w2.npz")
def _div_time_mean_for_wave(wave_k: int):
    ep1_k, ep2_k, div1_k, div2_k = ac.ComputeEPfluxDiv(
        lat, plev, u, v, t,
        wave=wave_k,
        do_ubar=True
    )
    # 时间平均后的总散度 (plev, lat)，单位 m/s/day
    return np.nanmean(div1_k + div2_k, axis=0)

div_w1 = _div_time_mean_for_wave(1)  # (plev, lat)
div_w2 = _div_time_mean_for_wave(2)

np.savez_compressed(
    cache_npz,
    lat=lat, plev=plev,
    div_w1=div_w1, div_w2=div_w2
)

print(f"✅ Cached: {cache_npz}")
print(f"lat size={lat.size}, plev size={plev.size}  (range {plev.max()}–{plev.min()} hPa)")


In [ ]:
cache_npz = "/home/weiji/restart_exam/plots/epflux_speedtest/cache/epflux_div_mean_100to1_w1w2.npz"
outdir    = "/home/weiji/restart_exam/plots/epflux_speedtest/cache"
os.makedirs(outdir, exist_ok=True)

# 选择：'w1' | 'w2' | 'sum'
mode = "sum"

Z = np.load(cache_npz)
lat  = Z["lat"]
plev = Z["plev"]
div_w1 = Z["div_w1"]  # (plev, lat), m/s/day
div_w2 = Z["div_w2"]

if mode == "w1":
    DIV = div_w1
    tag = "wave1"
elif mode == "w2":
    DIV = div_w2
    tag = "wave2"
else:
    DIV = div_w1 + div_w2
    tag = "wave1_2"

# 构造 DataArray：注意 dims 顺序为 ("plev","lat")，不要 transpose
DIV_da = xr.DataArray(DIV, coords={"plev": plev, "lat": lat}, dims=["plev","lat"])

# （可选）去掉最底层
max_plev = float(DIV_da.plev.max())
DIV_plot = DIV_da.sel(plev=DIV_da.plev.where(DIV_da.plev < max_plev, drop=True))

# 画图：x=lat（列），y=plev（行），Z.shape = (len(plev), len(lat))
levels = np.linspace(-4, 4, 17)
fig, ax = plt.subplots(figsize=(8,6))
cf = ax.contourf(DIV_plot.lat, DIV_plot.plev, DIV_plot, levels=levels, cmap="RdBu_r", extend="both")

ax.set_xlim([-90, 90])
ax.set_ylim([100, 1])
ax.set_yscale("log")
ax.set_xlabel("Latitude (°)")
ax.set_ylabel("Pressure (hPa)")
cbar = fig.colorbar(cf, ax=ax, pad=0.02)
cbar.set_label("EP-flux divergence [m/s/day]")


outfile = os.path.join(outdir, f"epflux_div_{tag}_ERA5_2015to2024_100to1hPa.png")
fig.savefig(outfile, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {outfile}")

In [ ]:
# --- Block E: Combine divergence (fill) + EP-flux vectors (arrows) on pressure coords (100–1 hPa) ---

# ==== 用户可调参数 ====
mode       = "sum"   # 'w1' | 'w2' | 'sum'
step_lat   = 10       # 纬向下采样步长（控制箭头疏密）
step_plev  = 1       # 垂直下采样步长
arrow_scale = 2e16   # 箭头缩放因子，数值越大箭头越短
# ====================

# 路径直接复用前面定义的 cache_dir
div_file = os.path.join(cache_dir, "epflux_div_mean_100to1_w1w2.npz")
vec_file = os.path.join(cache_dir, "epflux_mean_100to1_w1w2.npz")

# === 读取缓存 ===
D = np.load(div_file)
V = np.load(vec_file)
lat  = D["lat"];  plev = D["plev"]

# ---- 散度底图 (plev, lat) ----
if mode == "w1":
    DIV = D["div_w1"];  F1, F2 = V["ep1_w1"], V["ep2_w1"];  tag = "wave1"
elif mode == "w2":
    DIV = D["div_w2"];  F1, F2 = V["ep1_w2"], V["ep2_w2"];  tag = "wave2"
else:
    DIV = D["div_w1"] + D["div_w2"]
    F1  = V["ep1_w1"] + V["ep1_w2"]
    F2  = V["ep2_w1"] + V["ep2_w2"]
    tag = "wave1_2"

# 去掉最底层，防止近地面异常矢量
max_plev = float(plev.max())
mask = plev < max_plev
plev_plot = plev[mask]
DIV_plot  = DIV[mask, :]
F1_plot   = F1[mask, :]
F2_plot   = F2[mask, :]

# === 绘图 ===
fig, ax = plt.subplots(figsize=(8, 6))

# 1) 散度填色图
levels = np.linspace(-4, 4, 17)
cf = ax.contourf(lat, plev_plot, DIV_plot, levels=levels, cmap="RdBu_r", extend="both")

# 2) 矢量图（转换为 xarray 以适配 ac.PlotEPfluxArrows）
f1 = xr.DataArray(F1_plot, coords={"plev": plev_plot, "lat": lat}, dims=["plev", "lat"]).transpose("lat", "plev")
f2 = xr.DataArray(F2_plot, coords={"plev": plev_plot, "lat": lat}, dims=["plev", "lat"]).transpose("lat", "plev")

# 下采样
f1s = f1.isel(lat=slice(None, None, step_lat), plev=slice(None, None, step_plev))
f2s = f2.isel(lat=slice(None, None, step_lat), plev=slice(None, None, step_plev))

ac.PlotEPfluxArrows(
    f1s.lat, f1s.plev, f1s, f2s,
    fig, ax,
    xlim=[-90, 90],
    ylim=[100, 1],
    yscale="log",
    scale=arrow_scale  # ← 可调缩放参数
)

# 统一外观
ax.set_xlim([-90, 90])
ax.set_ylim([100, 1])
ax.set_yscale("log")
ax.set_xlabel("Latitude (°)")
ax.set_ylabel("Pressure (hPa)")
ax.set_title(f"EP-Flux vectors over divergence ({tag})  ERA5 2015–2024  100–1 hPa")

cbar = fig.colorbar(cf, ax=ax, pad=0.02)
cbar.set_label("EP-flux divergence [m/s/day]")

outfile = os.path.join(cache_dir, f"epflux_combined_{tag}_ERA5_2015to2024_100to1hPa.png")
fig.savefig(outfile, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {outfile}")


In [ ]:
# === Block A: EP-flux (wave 1 & 2) 1000–10 hPa — compute & cache ===

cache_npz_1000_10 = os.path.join(outdir, "epflux_mean_1000to10_w1w2.npz")

# 读取 ERA5，并截取 1000–10 hPa
ds_1000_10 = (
    xr.open_dataset(era5_file)
      .rename({"valid_time":"time", "pressure_level":"plev", "latitude":"lat", "longitude":"lon"})
      .sel(plev=slice(1000.0, 10.0))
)

lat_1000_10  = ds_1000_10.lat.values
plev_1000_10 = ds_1000_10.plev.values
u_1000_10    = ds_1000_10.u.values
v_1000_10    = ds_1000_10.v.values
t_1000_10    = ds_1000_10.t.values

def _compute_time_mean_for_wave_1000_10(k: int):
    ep1_k, ep2_k, *_ = ac.ComputeEPfluxDiv(lat_1000_10, plev_1000_10, u_1000_10, v_1000_10, t_1000_10,
                                           wave=k, do_ubar=True)
    return np.nanmean(ep1_k, axis=0), np.nanmean(ep2_k, axis=0)  # (plev, lat)

ep1_w1_1000_10, ep2_w1_1000_10 = _compute_time_mean_for_wave_1000_10(1)
ep1_w2_1000_10, ep2_w2_1000_10 = _compute_time_mean_for_wave_1000_10(2)

# 缓存：注意存的维度是 (plev, lat)
np.savez_compressed(
    cache_npz_1000_10,
    lat=lat_1000_10, plev=plev_1000_10,
    ep1_w1=ep1_w1_1000_10, ep2_w1=ep2_w1_1000_10,
    ep1_w2=ep1_w2_1000_10, ep2_w2=ep2_w2_1000_10
)

print(f"✅ Cached (1000–10 hPa): {cache_npz_1000_10}")
print(f"lat size={lat_1000_10.size}, plev size={plev_1000_10.size} (range {plev_1000_10.max()}–{plev_1000_10.min()} hPa)")
# === End Block A ===


In [ ]:
# === Plot Block B: 1000–10 hPa（参数风格与 100–1 hPa 块一致） ===

cache_npz = "/home/weiji/restart_exam/plots/epflux_speedtest/cache/epflux_mean_1000to10_w1w2.npz"
outdir    = "/home/weiji/restart_exam/plots/epflux_speedtest/cache"
os.makedirs(outdir, exist_ok=True)

# 选择绘制的模式：'w1' | 'w2' | 'sum'
# ⚠️ 如已在上一个块设定了 mode/step_lat/step_plev，可注释掉下面三行，避免重复设置
mode = "sum"

# 可选的下采样控制（仅影响显示密度，不改变结果）
step_lat  = 15
step_plev = 1

# 读取缓存
Z = np.load(cache_npz)
lat  = Z["lat"]
plev = Z["plev"]
ep1_w1, ep2_w1 = Z["ep1_w1"], Z["ep2_w1"]   # (plev, lat)
ep1_w2, ep2_w2 = Z["ep1_w2"], Z["ep2_w2"]

# 选择要画的场
if mode == "w1":
    ep1, ep2 = ep1_w1, ep2_w1
    tag = "wave1"
elif mode == "w2":
    ep1, ep2 = ep1_w2, ep2_w2
    tag = "wave2"
else:  # 'sum'
    ep1, ep2 = ep1_w1 + ep1_w2, ep2_w1 + ep2_w2
    tag = "wave1_2"

# 组织为 (lat, plev) 以适配 PlotEPfluxArrows
f1 = xr.DataArray(ep1, coords={"plev": plev, "lat": lat}, dims=["plev", "lat"]).transpose("lat","plev")
f2 = xr.DataArray(ep2, coords={"plev": plev, "lat": lat}, dims=["plev", "lat"]).transpose("lat","plev")

# 去掉最底层，避免近地面极端矢量影响展示
max_plev = float(f1.plev.max())  # 最大 hPa 即最低层
f1p = f1.sel(plev=f1.plev.where(f1.plev < max_plev, drop=True))
f2p = f2.sel(plev=f2.plev.where(f2.plev < max_plev, drop=True))

# 可选：横向/竖向稀疏（只影响显示）
f1p = f1p.isel(lat=slice(None,None,step_lat), plev=slice(None,None,step_plev))
f2p = f2p.isel(lat=slice(None,None,step_lat), plev=slice(None,None,step_plev))

# 绘图：1000–10 hPa，log-p
fig, ax = plt.subplots(figsize=(8,6))
ac.PlotEPfluxArrows(
    f1p.lat, f1p.plev,
    f1p, f2p,
    fig, ax,
    xlim=[-90, 90],
    ylim=[1000, 10],
    yscale="log",
    scale=5e15  # 与上一个块保持一致；如显拥挤可改 2e16 或 5e16
)
ax.set_xlabel("Latitude (°)")
ax.set_ylabel("Pressure (hPa)")
ax.set_title(f"EP-Flux ({tag})  ERA5 2015–2024  1000–10 hPa  do_ubar=True")

outfile = os.path.join(outdir, f"epflux_{tag}_ERA5_2015to2024_1000to10hPa.png")
fig.savefig(outfile, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {outfile}")
# === End Plot Block B ===
